# Research question:
To what extent does chronic disease burden improve the prediction of
continuous cognitive performance beyond demographic characteristics among older adults?

Research gap: Although chronic disease burden has been associated with poorer cognitive
performance and incorporated into predictive models of cognitive outcomes, its incremental
predictive value beyond demographic characteristics for continuous cognitive performance
remains insufficiently characterized.

# 1. Read the file

In [ ]:
from pathlib import Path
import pandas as pd

# Local path: Documents/BMEN 600
folder = Path.home() / "Documents" / "BMEN 600"
input_file = folder / "cleaned_latest_data.csv"

# Fallback to current directory or Google Colab environment path if the local path doesn't exist
if not input_file.exists():
    input_file = Path("/content/cleaned_latest_data.csv")
    if not input_file.exists():
        input_file = Path("cleaned_latest_data.csv")

data = pd.read_csv(
    input_file,
    dtype={"Participant ID": "string"},
    low_memory=False,
)

data.columns = data.columns.str.strip()

print(f"Records: {len(data):,}")
display(data.head())

Records: 13,053


,Participant ID,Survey year,Immediate recall,Delayed recall,Serial 7s,Backwards counting,Derived 0-27 cognition score,Hypertension,Diabetes,Cancer,...,Chronic Disease Burden,Age,Sex,Race/ethnicity,Education,Marital status,Smoking,BMI,Interview mode,Proxy respondent
0,010001010,2022,4.0,0.0,5.0,2.0,11.0,0.0,0.0,0.0,...,0.0,83.0,1,Non-Hispanic White,12.0,8.0,0.0,19.5,1.0,0.0
1,010003030,2022,5.0,2.0,4.0,2.0,13.0,1.0,0.0,0.0,...,5.0,66.0,2,Non-Hispanic White,16.0,7.0,0.0,36.0,1.0,0.0
2,010004040,2022,7.0,5.0,5.0,2.0,19.0,1.0,0.0,0.0,...,2.0,77.0,2,Non-Hispanic White,12.0,1.0,0.0,30.1,1.0,0.0
3,010013040,2022,5.0,6.0,5.0,2.0,18.0,1.0,0.0,0.0,...,1.0,74.0,2,Non-Hispanic White,13.0,5.0,0.0,25.2,1.0,0.0
4,010038010,2022,4.0,4.0,5.0,2.0,15.0,1.0,0.0,0.0,...,2.0,85.0,1,Non-Hispanic White,16.0,1.0,0.0,22.8,1.0,0.0


# 2.Redefine the numerical codes for each race category

In [ ]:
race_mapping = {
    "Non-Hispanic White": 0,
    "Non-Hispanic Other": 1,
    "Non-Hispanic Black": 2,
    "Hispanic": 3,
}

race = data["Race/ethnicity"].astype("string").str.strip()

invalid = race.isna() | ~race.isin(race_mapping)
if invalid.any():
    raise ValueError(
        f"Missing or unexpected race values: "
        f"{race.loc[invalid].unique().tolist()}"
    )

data["Race/ethnicity"] = race.map(race_mapping).astype("int64")

print("Race/ethnicity counts after recoding:")
print(data["Race/ethnicity"].value_counts().sort_index())

Race/ethnicity counts after recoding:
Race/ethnicity
0    8186
1     375
2    2616
3    1876
Name: count, dtype: int64


In [ ]:
print("Data types:")
print(data.dtypes)

non_numeric = data.select_dtypes(exclude="number").columns.tolist()
print("\nNon-numeric columns:", non_numeric)

model_columns = data.drop(columns=["Participant ID"])

print(
    "\nAll remaining columns are numeric:",
    model_columns.select_dtypes(exclude="number").empty,
)

Data types:
Participant ID                  string[python]
Survey year                              int64
Immediate recall                       float64
Delayed recall                         float64
Serial 7s                              float64
Backwards counting                     float64
Derived 0-27 cognition score           float64
Hypertension                           float64
Diabetes                               float64
Cancer                                 float64
Chronic lung disease                   float64
Heart disease                          float64
Stroke                                 float64
Arthritis                              float64
Chronic Disease Burden                 float64
Age                                    float64
Sex                                      int64
Race/ethnicity                           int64
Education                              float64
Marital status                         float64
Smoking                                float64
B

# 3.Pre-processing
## Define X1 and X2

In [ ]:
demographic_features = [ "Age", "Sex", "Race/ethnicity", "Education", "Marital status"]

health_features = ["Smoking","BMI"]

# Model 1: Demographics + health condition
x1_features = demographic_features + health_features

# Model 2: Demographics + health condition + Chronic Disease Burden
x2_features = x1_features + ["Chronic Disease Burden"]

X1 = data[x1_features].copy()
X2 = data[x2_features].copy()

print("Model 1 features:", X1.columns.tolist())
print("Model 1 shape:", X1.shape)

print("\nModel 2 features:", X2.columns.tolist())
print("Model 2 shape:", X2.shape)


Model 1 features: ['Age', 'Sex', 'Race/ethnicity', 'Education', 'Marital status', 'Smoking', 'BMI']
Model 1 shape: (13053, 7)

Model 2 features: ['Age', 'Sex', 'Race/ethnicity', 'Education', 'Marital status', 'Smoking', 'BMI', 'Chronic Disease Burden']
Model 2 shape: (13053, 8)


## Define Y

In [ ]:
# Outcome: Cognitive Score (0–27)
Y = data["Derived 0-27 cognition score"].copy()

print("Outcome:",Y.name)
print("Shape:", Y.shape)
print("Missing values:", Y.isna().sum())
print("Score range:", Y.min(), "to", Y.max())

assert X1.index.equals(Y.index)
assert X2.index.equals(Y.index)

Outcome: Derived 0-27 cognition score
Shape: (13053,)
Missing values: 0
Score range: 0.0 to 27.0


## Split the Data into Training and Test Sets

Use `train_test_split()` to divide the data into 70% training and 30% test sets.

- **X1:** Demographics + health condition (Model 1).
- **X2:** Demographics + health condition + Chronic Disease Burden (Model 2).
- **y:** Cognitive Score (0–27).

Split X1, X2, and y together so both models use the same participants in their training and test sets. Set `shuffle=True` to shuffle the records before splitting and `random_state=3` to make the split reproducible.

Use the training set to fit the models and the test set to evaluate and compare their predictive performance. Fit any preprocessing steps on the training set only, then apply them to the test set.

In [ ]:
from sklearn.model_selection import train_test_split

(X1_trainset,X1_testset,X2_trainset,X2_testset,Y_trainset,Y_testset,) = train_test_split(
    X1,X2,Y, test_size=0.3,random_state=3,shuffle=True,)

# Check that both models use the same records
assert X1_trainset.index.equals(X2_trainset.index)
assert X1_trainset.index.equals(Y_trainset.index)
assert X1_testset.index.equals(X2_testset.index)
assert X1_testset.index.equals(Y_testset.index)

print("X1 training set:", X1_trainset.shape)
print("X1 test set:", X1_testset.shape)
print("X2 training set:", X2_trainset.shape)
print("X2 test set:", X2_testset.shape)
print("y training set:", Y_trainset.shape)
print("y test set:", Y_testset.shape)

X1 training set: (9137, 7)
X1 test set: (3916, 7)
X2 training set: (9137, 8)
X2 test set: (3916, 8)
y training set: (9137,)
y test set: (3916,)


# Abbey: Random Forest
Create and train both Random Forest models

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def make_model(X):
    numeric_cols = X.select_dtypes(include='number').columns
    categorical_cols = X.select_dtypes(exclude='number').columns

    preprocessing = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ])

    return Pipeline([
        ('preprocessing', preprocessing),
        ('random_forest', RandomForestRegressor(
            n_estimators=100,
            random_state=3
        ))
    ])

# Create models using the correct training sets
model1 = make_model(X1_trainset)
model2 = make_model(X2_trainset)

# Train models using the correct training target
model1.fit(X1_trainset, Y_trainset)
model2.fit(X2_trainset, Y_trainset)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  Index(['Age', 'Sex', 'Race/ethnicity', 'Education', 'Marital status',
       'Smoking', 'BMI', 'Chronic Disease Burden'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index([], dtype='object'))])),
                ('random_forest', RandomForestRegressor(random_state=3))])

# Make predictions

In [ ]:
y_pred1 = model1.predict(X1_testset)
y_pred2 = model2.predict(X2_testset)

# Evaluate and compare the models

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

results = pd.DataFrame({
    'Model': ['Model 1', 'Model 2'],
    'MAE': [
        mean_absolute_error(Y_testset, y_pred1),
        mean_absolute_error(Y_testset, y_pred2)
    ],
    'RMSE': [
        mean_squared_error(Y_testset, y_pred1)**0.5,
        mean_squared_error(Y_testset, y_pred2)**0.5
    ],
    'R²': [
        r2_score(Y_testset, y_pred1),
        r2_score(Y_testset, y_pred2)
    ]
})

print(results)

     Model       MAE      RMSE        R²
0  Model 1  3.608676  4.553492  0.165991
1  Model 2  3.536984  4.449493  0.203653
